这一页把分块和 Embeddings 接起来，并把向量放进本地 Qdrant。Qdrant 负责保存向量和 payload，搜索时返回最相近的片段。

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
if not EMBEDDING_MODEL:
    raise RuntimeError("请在 .env 中配置 EMBEDDING_MODEL。")
model = SentenceTransformer(EMBEDDING_MODEL)

def split_markdown(text, max_chars=800):
    sections = re.split(r"\n(?=#{1,3}\s)", text)
    chunks, current = [], ""
    for section in sections:
        if current and len(current) + len(section) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += section + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for text in split_markdown(path.read_text(encoding="utf-8")):
            items.append({"source": str(path.relative_to(data_dir)), "text": text})
    return items

def build_index(chunks):
    vectors = model.encode([item["text"] for item in chunks], normalize_embeddings=True)
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="fashion_knowledge",
        vectors_config=models.VectorParams(size=vectors.shape[1], distance=models.Distance.COSINE),
    )
    client.upload_points(
        collection_name="fashion_knowledge",
        points=[models.PointStruct(id=i, vector=vector.tolist(), payload=chunk) for i, (vector, chunk) in enumerate(zip(vectors, chunks))],
    )
    return client

In [ ]:
chunks = load_chunks()
qdrant = build_index(chunks)
print(f"知识库片段：{len(chunks)}")

## 写一个最小搜索函数

In [ ]:
def search(question, top_k=4):
    question_vector = model.encode(question, normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name="fashion_knowledge",
        query=question_vector,
        limit=top_k,
    ).points
    return [{**hit.payload, "score": hit.score} for hit in hits]

for result in search("SKU-YG301 瑜伽裤的面料成分是什么？"):
    print(f"{result['score']:.3f}  {result['source']}")
    print(result["text"][:160].replace("\n", " ") + "...")

## 复现 P09：货号查询的排名实验

P09 的结论不能靠手写一个固定名次。下面固定模型、分块规则、知识库和查询，直接报告目标片段在纯向量检索中的真实排名。

如果目标片段没有进入 Top-K，就能看到“业务上唯一的货号”并不等于“向量空间里一定排第一”。

In [ ]:
def rank_of_target(question, target_source, target_keyword):
    question_vector = model.encode(question, normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name="fashion_knowledge", query=question_vector, limit=len(chunks)
    ).points
    target_rank = next(
        rank for rank, hit in enumerate(hits, start=1)
        if hit.payload["source"] == target_source and target_keyword in hit.payload["text"]
    )
    return target_rank, hits

queries = [
    "SKU-JK902",
    "SKU-JK902 Cordura 500D",
    "冲锋衣的高强度耐磨面料",
]
target_source = "产品/冲锋衣-JK902/产品规格.md"
for query in queries:
    rank, hits = rank_of_target(query, target_source, "SKU-JK902")
    print(f"查询：{query} | 目标片段排名：{rank}/{len(hits)}")
    for hit in hits[:5]:
        print(f"  {hit.score:.3f}  {hit.payload['source']}")
    print()

## 主题相似，不代表答案可以互换

向量相似度反映的是主题接近程度。下面两个片段都在讨论尺码，但结论相反。

In [ ]:
size_texts = [
    "这款女式瑜伽裤版型偏大，建议通常尺码的顾客选择小一码。",
    "这款女式瑜伽裤版型偏小，建议通常尺码的顾客选择大一码。",
]
size_query = "这款瑜伽裤尺码怎么选？"
size_vectors = model.encode(size_texts, normalize_embeddings=True)
size_query_vector = model.encode(size_query, normalize_embeddings=True)
size_scores = size_vectors @ size_query_vector
for text, score in sorted(zip(size_texts, size_scores), key=lambda item: item[1], reverse=True):
    print(f"{score:.3f}  {text}")

print("相似度高说明都与尺码主题相关，但不能据此判断应该买大一码还是小一码。")

## 再看相似型号

下面是课堂构造的商品名。查询 `iPhone Pro4` 时，三个名称都属于相近主题，但只有一个是准确实体。

In [ ]:
catalog_texts = [
    "商品名称：iPhone Pro4；型号：IPHONE-PRO4；定位：专业影像手机。",
    "商品名称：苹果 Pro4；型号：APPLE-PRO4；定位：专业影像手机。",
    "商品名称：苹果 Plug5；型号：APPLE-PLUG5；定位：便携手机配件。",
]
catalog_query = "iPhone Pro4"
catalog_vectors = model.encode(catalog_texts, normalize_embeddings=True)
catalog_query_vector = model.encode(catalog_query, normalize_embeddings=True)
catalog_scores = catalog_vectors @ catalog_query_vector
for text, score in sorted(zip(catalog_texts, catalog_scores), key=lambda item: item[1], reverse=True):
    print(f"{score:.3f}  {text}")

print("这里的相似度只能说明主题接近，准确型号仍需要精确词项确认。")